In [ ]:
!pip install opencv-python

In [ ]:

import cv2
import numpy as np
import os
from google.colab import files
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')

print("Mounted. Ready.")


input_dir = "/content/drive/MyDrive/BAD MANGOES1"

output_dir = "/content/drive/MyDrive/mango_masks_S1/"
os.makedirs(output_dir, exist_ok=True)

print("Input:", input_dir)
print("Output:", output_dir)



def generate_mask(image_path, k=4):
    img = cv2.imread(image_path)

    if img is None:
        print("Error reading:", image_path)
        return None


    img = cv2.resize(img, (512, 512))


    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


    pixel_values = gray.reshape((-1, 1)).astype(np.float32)

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, 10,
                                    cv2.KMEANS_RANDOM_CENTERS)

    centers = np.uint8(centers)
    segmented = centers[labels.flatten()].reshape(gray.shape)

    n
    damaged_cluster = np.argmin(centers)

    mask = np.zeros_like(gray)
    mask[segmented == centers[damaged_cluster]] = 255

    return mask


image_files = [f for f in os.listdir(input_dir)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print("Found", len(image_files), "images.")

for file in tqdm(image_files):
    img_path = os.path.join(input_dir, file)
    mask = generate_mask(img_path, k=4)

    if mask is not None:
        out_path = os.path.join(output_dir, file.replace(".jpg", ".png")
                                             .replace(".jpeg", ".png"))
        cv2.imwrite(out_path, mask)

print("Done! Masks saved to:", output_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted. Ready.
Input: /content/drive/MyDrive/BAD MANGOES1
Output: /content/drive/MyDrive/mango_masks_S1/
Found 1500 images.


100%|██████████| 1500/1500 [48:41<00:00,  1.95s/it]

Done! Masks saved to: /content/drive/MyDrive/mango_masks_S1/


In [ ]:
import cv2
import os

input_dir = "/content/drive/MyDrive/mango_masks_S1"
output_dir = "/content/drive/MyDrive/mango_masks_S1_resized_512/"
os.makedirs(output_dir, exist_ok=True)

TARGET_SIZE = (512, 512)

for file in os.listdir(input_dir):
    if file.lower().endswith(('.png', '.jpeg', '.jpg', '.bmp', '.tif', '.tiff')):


        img_path = os.path.join(input_dir, file)
        img = cv2.imread(img_path)

        if img is None:
            print("Error reading:", file)
            continue


        img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

        out_name = os.path.splitext(file)[0] + ".jpg"
        out_path = os.path.join(output_dir, out_name)


        cv2.imwrite(out_path, img_resized, [cv2.IMWRITE_JPEG_QUALITY, 95])

print("Done! All images resized to 512×512 JPG.")


Done! All images resized to 512×512 JPG.
